[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tunnel-ai/way/blob/main/notebooks/06_01_main_classical.ipynb)

# Module 6, Vision: The Pre-Deep-Learning Baseline

**Notebook:** `06_01_main_classical`

## What we're doing

Before CNNs, image classification looked like every other tabular ML problem: extract a fixed set of **hand-engineered features** from each image, stack them into a `(n_samples, n_features)` matrix, and feed that to a logistic regression or a random forest. People built whole careers around designing better features (SIFT, HOG, SURF, color moments).

We do that recipe end-to-end on **EuroSAT**, a small satellite-imagery dataset with 10 land-cover classes. By the end of this notebook we'll have a classical-CV baseline. The next two notebooks then beat it...well...maybe?

## The recipe

| Step | Tool | What it does |
| --- | --- | --- |
| Load | Zenodo + PIL | grab EuroSAT (10-class, 64x64 satellite RGB) into numpy |
| Engineer | numpy + scipy | per-channel color stats + color histograms + Sobel edge density |
| Train | scikit-learn | logistic regression and random forest on the feature table |
| Evaluate | confusion matrix, top mistakes | see what classical features can and can't separate |

Every feature in this notebook is something a human picked *and* named. That's the whole point of the contrast with [`06_02_main_cnn.ipynb`](06_02_main_cnn.ipynb), where the network *learns* its own features.

## 0) Setup

Standard scientific Python plus PIL (ships with most environments). The first run downloads EuroSAT (~90 MB) from Zenodo into `assets/data/`. Re-runs are instant.

> Why not TensorFlow Datasets? `tfds.load('eurosat/rgb')` points at a host that often returns 403. Going direct to Zenodo is more reliable and adds zero dependencies. But pick your favorite

In [ ]:
import os, zipfile

import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt
from PIL import Image
from scipy.ndimage import convolve

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

RNG = np.random.default_rng(1955)

## 1) Load EuroSAT and pull a manageable subsample

EuroSAT has 27,000 images. We don't need all of them, hand-engineered features and a tabular model run in seconds on a few thousand examples, and that's enough to show what classical CV can and can't do.

We pull a random 5,000-image sample, then split 80/20 (stratified) into train and test. Stratification matters: EuroSAT is *not* perfectly balanced (some classes have 2,500 images, others 2,000), and we want both splits to see every class.

In [ ]:
DATA_DIR = "assets/data"
ZIP_PATH = f"{DATA_DIR}/EuroSAT_RGB.zip"
EXTRACT_DIR = f"{DATA_DIR}/EuroSAT_RGB"
URL = "https://zenodo.org/records/7711810/files/EuroSAT_RGB.zip"

os.makedirs(DATA_DIR, exist_ok=True)
if not os.path.exists(ZIP_PATH):
    print("Downloading EuroSAT (~90 MB, one-time)...")
    with requests.get(URL, stream=True, timeout=120) as r:
        r.raise_for_status()
        with open(ZIP_PATH, "wb") as f:
            for chunk in r.iter_content(chunk_size=1 << 20):
                f.write(chunk)
if not os.path.exists(EXTRACT_DIR):
    with zipfile.ZipFile(ZIP_PATH) as z:
        z.extractall(EXTRACT_DIR)

# The zip layout has a single top-level folder containing the class subfolders.
# Find the directory that holds the 10 class folders.
def find_class_dir(root):
    for r, dirs, _ in os.walk(root):
        if len(dirs) >= 8:
            return r
    raise RuntimeError(f"Could not find class folders under {root}")

CLASS_DIR = find_class_dir(EXTRACT_DIR)
label_names = sorted(os.listdir(CLASS_DIR))
num_classes = len(label_names)
print("Classes:", label_names)

# Collect (path, label) pairs across all classes, then sample N_SAMPLES of them.
N_SAMPLES = 5000
all_paths = []
for ci, cls in enumerate(label_names):
    for fn in os.listdir(os.path.join(CLASS_DIR, cls)):
        all_paths.append((os.path.join(CLASS_DIR, cls, fn), ci))
RNG.shuffle(all_paths)
sampled = all_paths[:N_SAMPLES]

X_img = np.zeros((N_SAMPLES, 64, 64, 3), dtype=np.uint8)
y     = np.zeros(N_SAMPLES, dtype=np.int64)
for i, (path, lab) in enumerate(sampled):
    X_img[i] = np.asarray(Image.open(path).convert("RGB"))
    y[i] = lab

print("Image array:", X_img.shape, X_img.dtype)
print("Sampled class counts:", dict(zip(*np.unique(y, return_counts=True))))

### Look at the data

EuroSAT classes look very different from each other to a human (forest is green, sea is blue, residential is gridded). A human eye is solving this with thousands of learned features. Our classical pipeline is going to try with about 35.

In [ ]:
fig, axs = plt.subplots(2, 5, figsize=(12, 5))
for c, ax in zip(range(num_classes), axs.flat):
    i = np.where(y == c)[0][0]
    ax.imshow(X_img[i])
    ax.set_title(label_names[c], fontsize=10)
    ax.axis("off")
plt.tight_layout()
plt.show()

## 2) Engineer features

Three feature blocks, each capturing a different signal:

1. **Per-channel color statistics** (6 features). Mean and standard deviation of red, green, blue. The simplest possible color descriptor. *Forest* will have a high green mean and low variance; *Sea/Lake* will have a high blue mean.
2. **Per-channel color histograms** (24 features). Each channel binned into 8 buckets. Captures the *distribution* of color, not just the average. *Highway* (gray asphalt + green grass) and *PermanentCrop* (uniform green) have similar means but different shapes.
3. **Edge density** (4 features). Mean absolute Sobel response (x and y), separately for the top and bottom halves of the image. Gives a coarse sense of how 'busy' the image is. *Residential* has lots of straight-line edges; *Forest* has noisy short edges; *Sea* has almost none.

**34 features per image, all named, all interpretable.** That's the honest classical-CV trade: every column is something a human chose.

In [ ]:
def color_stats(img_batch):
    """Per-channel mean and std. Returns (N, 6)."""
    x = img_batch.astype(np.float32) / 255.0
    means = x.mean(axis=(1, 2))                     # (N, 3)
    stds  = x.std(axis=(1, 2))                      # (N, 3)
    return np.concatenate([means, stds], axis=1)

feat_color = color_stats(X_img)
print("color_stats shape:", feat_color.shape)

In [ ]:
def color_histograms(img_batch, bins=8):
    """Per-channel histograms, normalized to a probability. Returns (N, 3*bins)."""
    N = img_batch.shape[0]
    out = np.zeros((N, 3 * bins), dtype=np.float32)
    edges = np.linspace(0, 256, bins + 1)
    for i in range(N):
        for c in range(3):
            h, _ = np.histogram(img_batch[i, ..., c], bins=edges)
            out[i, c * bins : (c + 1) * bins] = h / h.sum()
    return out

feat_hist = color_histograms(X_img, bins=8)
print("color_histograms shape:", feat_hist.shape)

In [ ]:
SOBEL_X = np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], dtype=np.float32)
SOBEL_Y = SOBEL_X.T

def edge_density(img_batch):
    """Mean abs Sobel response on the grayscale image, split top/bottom half.
    Returns (N, 4): [top_x, top_y, bot_x, bot_y]."""
    gray = img_batch.astype(np.float32).mean(axis=3) / 255.0   # (N, 64, 64)
    out = np.zeros((gray.shape[0], 4), dtype=np.float32)
    half = gray.shape[1] // 2
    for i in range(gray.shape[0]):
        gx = np.abs(convolve(gray[i], SOBEL_X, mode="nearest"))
        gy = np.abs(convolve(gray[i], SOBEL_Y, mode="nearest"))
        out[i] = [gx[:half].mean(), gy[:half].mean(),
                  gx[half:].mean(), gy[half:].mean()]
    return out

feat_edges = edge_density(X_img)
print("edge_density shape:", feat_edges.shape)

### Stack into a tabular table

Concat the three feature blocks. Now we have a regular `(n_samples, n_features)` matrix, the kind of thing every notebook in Modules 1-4 was built around. Image classification just became a tabular ML problem.

In [ ]:
X = np.concatenate([feat_color, feat_hist, feat_edges], axis=1)

feat_names = (
    [f"mean_{c}" for c in "rgb"] + [f"std_{c}" for c in "rgb"]
    + [f"hist_{c}_{b}" for c in "rgb" for b in range(8)]
    + ["edge_top_x", "edge_top_y", "edge_bot_x", "edge_bot_y"]
)
print("Feature matrix:", X.shape)
pd.DataFrame(X[:5], columns=feat_names).round(3)

## 3) Train/test split and standardize

Standard 80/20 stratified split. We z-score the features so logistic regression and random forest see comparable scales (RF doesn't strictly need it, but it costs nothing).

In [ ]:
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=1955
)
scaler = StandardScaler().fit(X_tr)
X_tr_s = scaler.transform(X_tr)
X_te_s = scaler.transform(X_te)
print(f"train: {X_tr_s.shape}   test: {X_te_s.shape}")

## 4) Train two classical baselines

**Logistic regression** as the linear baseline (each feature has one coefficient per class, easy to inspect). **Random forest** as the non-linear baseline (handles feature interactions automatically, classic strong tabular default).

Random chance on 10 classes is 10%. Anything above that is the features doing real work.

In [ ]:
lr = LogisticRegression(max_iter=2000, C=1.0)
lr.fit(X_tr_s, y_tr)
lr_acc = accuracy_score(y_te, lr.predict(X_te_s))

rf = RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=1955)
rf.fit(X_tr_s, y_tr)
rf_acc = accuracy_score(y_te, rf.predict(X_te_s))

print(f"Logistic regression test accuracy: {lr_acc:.3f}")
print(f"Random forest       test accuracy: {rf_acc:.3f}")
print(f"Random chance baseline           : {1/num_classes:.3f}")

### Per-class breakdown (random forest)

(everything is better with random forest as we know).

Aggregate accuracy hides which classes were easy and which were hard. The classes with strong color signatures (Forest, SeaLake) score high; the ones that look alike at 64x64 (the various crop classes) all kind of blur together.

In [ ]:
y_pred_rf = rf.predict(X_te_s)
print(classification_report(y_te, y_pred_rf, target_names=label_names))

### Confusion matrix

The bright off-diagonal entries are the meaningful confusions. Crop classes melt into each other; built-up classes (Highway / Industrial / Residential) confuse each other. So maybe that's a *real* limitation of the features we built: color and edge density don't capture the spatial layout that distinguishes 'highway' from 'industrial'.

In [ ]:
cm = confusion_matrix(y_te, y_pred_rf)
fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(num_classes)); ax.set_xticklabels(label_names, rotation=90, fontsize=9)
ax.set_yticks(range(num_classes)); ax.set_yticklabels(label_names, fontsize=9)
ax.set_xlabel("predicted"); ax.set_ylabel("true")
ax.set_title("Random forest confusion matrix")
for i in range(num_classes):
    for j in range(num_classes):
        ax.text(j, i, cm[i, j], ha="center", va="center",
                color="white" if cm[i, j] > cm.max() / 2 else "black", fontsize=8)
plt.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout()
plt.show()

### High-confidence mistakes

When the random forest is wrong *and* it's confident, those are the cases where our features actively misled the model. Useful diagnostic: look at the picture, ask 'would I have gotten this right with only these 34 features'?

In [ ]:
probs = rf.predict_proba(X_te_s)
preds = probs.argmax(axis=1)
confs = probs.max(axis=1)
test_idx = np.arange(len(y_te))
mistakes = test_idx[preds != y_te]
mistakes = mistakes[np.argsort(-confs[mistakes])][:8]

# We need the original images for the test split, so reconstruct from the train_test_split
_, X_img_te, _, _ = train_test_split(X_img, y, test_size=0.2, stratify=y, random_state=1955)

fig, axs = plt.subplots(2, 4, figsize=(12, 6))
for k, ax in zip(mistakes, axs.flat):
    ax.imshow(X_img_te[k])
    ax.set_title(
        f"true: {label_names[y_te[k]]}\npred: {label_names[preds[k]]} ({confs[k]:.2f})",
        fontsize=9,
    )
    ax.axis("off")
plt.tight_layout()
plt.show()

## What we built

From raw 64x64 satellite tiles to a working classifier:

- **34 hand-named features** per image (color stats + color histograms + edge density),
- a **logistic regression** and a **random forest** trained on those features,
- a **per-class report** and a **confusion matrix** showing where the features earn their keep and where they fail.

Two things about this baseline:

1. **It's not zero.** Color + edges already separate forest from water, residential from open land, and the trees handle the obvious non-linearities. This was the state of the art for years.
2. **It's not great.** The crop classes confuse each other. The built-up classes confuse each other. The features we picked don't capture *spatial structure* well, things like 'gridded layout' or 'long parallel lines' need much more sophisticated descriptors (HOG, SIFT, GLCM textures), and even those have a ceiling.

## Where the next notebook goes

[`06_02_main_cnn.ipynb`](06_02_main_cnn.ipynb) keeps the same dataset and the same classification task, but stops hand-designing features. A CNN learns its own filters from the data, and on this dataset it produces a noticeably better baseline with no human in the feature-engineering loop.

## What can you play with?

- **`N_SAMPLES`**, more data helps the random forest more than the logistic regression.
- **`bins`** in `color_histograms`, finer bins give more resolution but also more noise.
- **add HOG** via `skimage.feature.hog` for a much stronger edge/shape descriptor (this is what the field actually used).
- **try gradient boosting** (`HistGradientBoostingClassifier`); it usually beats RF by a few points on this kind of feature table.